# QSCI: quantum-selected configuration interaction

QSCI samples a prepared state in the computational basis, keeps the most frequently observed electron configurations, and diagonalises the Hamiltonian classically in the subspace they span ([Kanno et al., arXiv:2302.11320](https://arxiv.org/abs/2302.11320)). Two kinds of input state are shown: a VQE-optimised UCCSD state, and Hartree-Fock evolved in time to several points and pooled, the time-averaged TE-QSCI of [Mikkelsen and Nakagawa, arXiv:2412.13839](https://arxiv.org/abs/2412.13839).

The system is N$_2$ with its bond stretched to 2.5 Å, well beyond the 1.10 Å equilibrium. Breaking the triple bond makes a single determinant a poor description, so the ground state spreads over many configurations. A 6-electron, 6-orbital active space gives a 12-qubit problem.

In [ ]:
from pyscf import gto, mcscf, scf

from qarp.operators import JordanWigner
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.pyscf import active_space_from_mf

bond_length = 2.5  # Å; equilibrium is about 1.10 Å
mol = gto.M(atom=f"N 0 0 0; N 0 0 {bond_length}", basis="sto-3g", verbose=0)
mf = scf.RHF(mol).run()
assert mf.converged

n_active_electrons, n_active_orbitals = 6, 6
integrals, hf_onv = active_space_from_mf(mf, n_active_electrons, n_active_orbitals)
hamiltonian = restricted_integrals_to_fermion_operator(*integrals)
n_qubits = len(hf_onv)
n_electrons = (sum(hf_onv[0::2]), sum(hf_onv[1::2]))  # (n_alpha, n_beta); abab order

e_hf = mf.e_tot
e_casci = mcscf.CASCI(mf, n_active_orbitals, n_active_electrons).kernel()[0]
print(f"{n_qubits} qubits, sector (n_alpha, n_beta) = {n_electrons}")
print(f"Hartree-Fock  {e_hf:.6f} Ha")
print(f"CASCI         {e_casci:.6f} Ha   (Hartree-Fock is {e_hf - e_casci:.3f} Ha above)")

Two limits check the setup before any sampling. A distribution holding only the Hartree-Fock determinant must reproduce the Hartree-Fock energy, and one covering every configuration in the sector must reproduce CASCI.

In [ ]:
import itertools

from qarp.algorithms import QSCI

hf_only = QSCI(hamiltonian, distribution={tuple(hf_onv): 1.0}, n_electrons=n_electrons)
e_hf_only = hf_only.build().run()[0][0]

sector = [
    bits
    for bits in itertools.product((0, 1), repeat=n_qubits)
    if (sum(bits[0::2]), sum(bits[1::2])) == n_electrons
]
complete = QSCI(
    hamiltonian, distribution={bits: 1 / len(sector) for bits in sector}, n_electrons=n_electrons
)
e_complete = complete.build().run()[0][0]

print(f"Hartree-Fock determinant only: {e_hf_only - e_hf:+.1e} Ha from Hartree-Fock")
print(f"All {len(sector)} configurations:     {e_complete - e_casci:+.1e} Ha from CASCI")
assert abs(e_hf_only - e_hf) < 1e-8
assert abs(e_complete - e_casci) < 1e-8

## A VQE-prepared input state

QSCI samples any prepared state. Here a UCCSD ansatz is optimised with VQE and the optimised state is sampled. `get_final_state_block()` returns it with the parameters bound, so it passes straight to QSCI.

In [ ]:
import numpy as np

from qarp.algorithms import VQE, Sampler
from qarp.blocks import CompositeBlock, MappedONVStateBlock, UCCBlock
from qarp.engines import QarpEngine

qubit_hamiltonian = JordanWigner().encode_operator(hamiltonian)
ansatz = CompositeBlock([MappedONVStateBlock(hf_onv, JordanWigner()), UCCBlock(hf_onv)])
ansatz.build()
vqe = VQE(
    qubit_hamiltonian,
    ket=ansatz,
    initial_parameters=np.zeros(len(ansatz.symbols)),
    gradient=True,
)
vqe.build()
e_vqe, _ = vqe.run()

vqe_qsci = QSCI(
    hamiltonian,
    ket=vqe.get_final_state_block(),
    primitive=Sampler(n_shots=2000),
    engine=QarpEngine(seed=42),
    n_electrons=n_electrons,
).build()
vqe_energies, _ = vqe_qsci.run()

print(f"VQE energy      {e_vqe:.6f} Ha   ({e_vqe - e_casci:.1e} Ha above CASCI)")
print(f"QSCI energy     {vqe_energies[0]:.6f} Ha   ({vqe_energies[0] - e_casci:.1e} Ha above CASCI)")
print(f"subspace        {len(vqe_qsci.configurations)} of {len(sector)} configurations")
print(f"in-sector mass  {vqe_qsci.in_sector_mass:.4f}")
assert e_casci - 1e-8 <= vqe_energies[0]

## Time-evolved input states

Each block prepares Hartree-Fock and applies a Trotterised $e^{-iHt}$. QSCI samples every block with the same number of shots and pools the results with equal weight, which is the time average for equally spaced times.

Some samples land outside the $(n_\alpha, n_\beta)$ sector, because splitting $H$ into separate Pauli exponentials breaks particle-number conservation slightly. QSCI warns about them, excludes them from selection, and keeps them on `qsci.rejected`.

In [ ]:
from qarp.blocks import TrotterBlock

times = [0.5, 1.0, 1.5, 2.0]
blocks = [
    CompositeBlock(
        [
            MappedONVStateBlock(hf_onv, JordanWigner()),
            TrotterBlock(n_qubits, qubit_hamiltonian, steps=1, time=t),
        ]
    )
    for t in times
]

qsci = QSCI(
    hamiltonian,
    ket=blocks,
    primitive=Sampler(n_shots=2000),
    engine=QarpEngine(seed=42),
    n_electrons=n_electrons,
).build()
energies, coefficients = qsci.run()

print(f"TE-QSCI energy  {energies[0]:.6f} Ha")
print(f"error vs CASCI  {energies[0] - e_casci:.1e} Ha")
print(f"subspace        {len(qsci.configurations)} of {len(sector)} configurations")
print(f"in-sector mass  {qsci.in_sector_mass:.4f}")
assert e_casci - 1e-8 <= energies[0] < e_casci + 5e-3

## Samples gathered elsewhere

QSCI also accepts samples it did not produce, through `distribution=`: the route for hardware results, or for any sampling done outside QSCI. Here the Trotter blocks from the previous section are sampled directly, one `Sampler` per time point, and the results are merged outside QSCI. With equal shots per time point the merged pool is the plain mean of the per-time distributions, the same time average QSCI forms from `ket=`, so the two energies must agree.

In [ ]:
samplers = [Sampler(ket=block, n_shots=2000) for block in blocks]
engine = QarpEngine(seed=42)
engine.build(samplers)
per_time = engine.run()  # one {bits: probability} distribution per time point

merged = {}
for distribution in per_time:
    for bits, probability in distribution.items():
        merged[bits] = merged.get(bits, 0.0) + probability / len(per_time)

external = QSCI(hamiltonian, distribution=merged, n_electrons=n_electrons).build()
external_energies, _ = external.run()

print(f"merged externally, distribution=  {external_energies[0]:.10f} Ha")
print(f"merged by QSCI, ket=blocks        {qsci.energies[0]:.10f} Ha")
assert abs(external_energies[0] - qsci.energies[0]) < 1e-10

## Convergence with subspace size

QSCI is usually reported as energy against subspace size, and that curve only means something if every point is selected from the same samples. `qsci.distribution` holds the pooled samples, so each point below re-selects from it without running a circuit.

In [ ]:
import warnings

import matplotlib.pyplot as plt

n_sampled = len(qsci.configurations)
floor = qsci.energies[0] - e_casci  # every sampled configuration included
sizes, errors = [], []
for size in range(5, n_sampled + 5, 5):
    size = min(size, n_sampled)
    point = QSCI(
        hamiltonian,
        distribution=qsci.distribution,
        n_electrons=n_electrons,
        max_subspace_dim=size,
    ).build()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)  # the same leakage, reported above
        error = point.run()[0][0] - e_casci
    sizes.append(size)
    errors.append(error)
    if error - floor <= max(0.05 * floor, 1e-6):
        break  # within 5% of the floor: flat on a log axis from here on

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.semilogy(sizes, errors, marker="o", linewidth=2, markersize=6)
ax.axhline(1.6e-3, linestyle="--", linewidth=1, color="0.5")
ax.annotate(
    "chemical accuracy",
    xy=(sizes[0], 1.6e-3),
    xytext=(0, 4),
    textcoords="offset points",
    ha="left",
    va="bottom",
    color="0.35",
)
ax.set_title("TE-QSCI error against CASCI, one sample set")
ax.set_xlabel("max_subspace_dim")
ax.set_ylabel("E − E_CASCI (Ha)")
ax.grid(True, color="0.9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

# Nested subspaces: the energy can only fall as the cap rises.
assert all(later <= earlier + 1e-12 for earlier, later in zip(errors, errors[1:], strict=False))